# S04.1 – Утечка через preprocessing: плохой vs хороший протокол

Цель: показать, как метрики улетают, если сделать `.fit` на всём датасете до split.

## Что вы освоите
- Понять правило: любой preprocessing с `.fit` обучается только на train
- Увидеть разницу между 'плохим' и 'хорошим' протоколом на цифрах
- Освоить Pipeline как стандартный способ предотвратить утечки

## Важные оговорки
- Мы не изучаем KNN как модельную тему. Используем её как инструмент, чувствительный к масштабу признаков.
- Главное – протокол.

---


## Среда, воспроизводимость и стиль эксперимента

Перед кодом – несколько правил, которые будут повторяться во всех ноутбуках:

1) **Воспроизводимость**: фиксируем `random_state` / seed.  
2) **Разделение данных**: test‑часть – это *священная зона*. Мы смотрим на неё только в конце.  
3) **Всё, что "обучается" (`.fit`)** должно видеть только train‑данные (иначе легко получить утечки).  
4) **Sanity‑checks**: после каждого шага проверяем, что получился ожидаемый результат (формы, распределения, пересечения и т.д.).


In [5]:
# Импорты: минимальный, но достаточный набор
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# Для красивых картинок (простая визуализация)
import matplotlib.pyplot as plt

# Фиксируем seed для воспроизводимости
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("numpy:", np.__version__)
print("pandas:", pd.__version__)


numpy: 2.0.2
pandas: 2.2.2


## Общие функции (оценка моделей и печать метрик)

Чтобы не копировать одно и то же вручную, заведём пару функций.

Важно: эти функции *ничего не делают магически*. Мы специально пишем их максимально прозрачно,
чтобы вы видели, какие именно метрики считаются и на каких данных.


In [6]:
def summarize_binary_metrics(y_true, y_pred, *, positive_label=1):
    """Считает базовые метрики бинарной классификации.

    Мы считаем:
    - accuracy: доля верных ответов
    - precision: насколько "чистые" наши позитивные предсказания
    - recall: насколько хорошо мы находим позитивный класс
    - f1: гармоническое среднее precision и recall

    Почему это важно: в задачах безопасности цена FP и FN может быть разной.
    """
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, pos_label=positive_label, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, pos_label=positive_label, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, pos_label=positive_label, zero_division=0)),
    }

def print_confusion(y_true, y_pred, labels=(0,1)):
    """Печать матрицы ошибок и пояснения, что есть что."""
    cm = confusion_matrix(y_true, y_pred, labels=list(labels))
    df = pd.DataFrame(cm, index=[f"true_{l}" for l in labels], columns=[f"pred_{l}" for l in labels])
    display(df)
    tn, fp, fn, tp = cm.ravel()
    print(f"TN={tn} (верно: 0), FP={fp} (ложная тревога), FN={fn} (пропуск), TP={tp} (верно: 1)")
    return cm

def evaluate_model(model, X_train, y_train, X_test, y_test, *, model_name="model"):
    """Обучает модель на train и оценивает на test. Возвращает словарь метрик."""
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    metrics = summarize_binary_metrics(y_test, y_pred)
    print(f"=== {model_name} ===")
    print(metrics)
    print("Confusion matrix:")
    _ = print_confusion(y_test, y_pred)
    return metrics


## Почему именно preprocessing может давать утечку

Типичная ошибка:
1) "Давайте отнормируем данные" → делаем `scaler.fit_transform(X)` на *всём* X.
2) Потом делаем split на train/test.
3) Получаем завышенные метрики.

Почему это утечка:
- scaler "увидел" распределение test‑данных и подстроил параметры (среднее/стандартное отклонение).
- это не так критично, как target leakage, но это всё равно "подсматривание".

Правильный подход:
- split сначала,
- scaler.fit только на train,
- scaler.transform на test,
или, лучше, `Pipeline`.


## Данные

Возьмём `breast_cancer` и превратим в numpy‑массив.


In [7]:
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer(as_frame=True)
X = data.data.values
y = data.target.values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
print(X_train.shape, X_test.shape)


(426, 30) (143, 30)


## Модель: KNN (чувствительна к масштабу)

KNN сравнивает расстояния между объектами. Если один признак измеряется в больших числах,
он доминирует над остальными. Поэтому нормализация (scaling) важна.

Мы используем KNN только как "датчик" того, что preprocessing влияет на качество.


In [8]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

knn = KNeighborsClassifier(n_neighbors=7)


## Плохой протокол (утечка): scaler на всём датасете → split уже после

Мы специально делаем неправильно, чтобы увидеть эффект.


In [9]:
scaler_bad = StandardScaler()
X_scaled_all = scaler_bad.fit_transform(X)  # ❌ УТЕЧКА: fit на всём X, включая будущий test

X_train_bad, X_test_bad, y_train_bad, y_test_bad = train_test_split(
    X_scaled_all, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

knn_bad = KNeighborsClassifier(n_neighbors=7)
knn_bad.fit(X_train_bad, y_train_bad)
y_pred_bad = knn_bad.predict(X_test_bad)

bad = summarize_binary_metrics(y_test_bad, y_pred_bad)
print("BAD protocol metrics:", bad)
print_confusion(y_test_bad, y_pred_bad)


BAD protocol metrics: {'accuracy': 0.972027972027972, 'precision': 0.967391304347826, 'recall': 0.9888888888888889, 'f1': 0.978021978021978}


,pred_0,pred_1
true_0,50,3
true_1,1,89


TN=50 (верно: 0), FP=3 (ложная тревога), FN=1 (пропуск), TP=89 (верно: 1)


array([[50,  3],
       [ 1, 89]])

## Хороший протокол: split → scaler.fit(train) → scaler.transform(test)

То же самое, но правильно (fit только на train).


In [10]:
scaler_good = StandardScaler()
X_train_scaled = scaler_good.fit_transform(X_train)   # ✅ fit только на train
X_test_scaled  = scaler_good.transform(X_test)

knn_good = KNeighborsClassifier(n_neighbors=7)
knn_good.fit(X_train_scaled, y_train)
y_pred_good = knn_good.predict(X_test_scaled)

good = summarize_binary_metrics(y_test, y_pred_good)
print("GOOD protocol metrics:", good)
print_confusion(y_test, y_pred_good)


GOOD protocol metrics: {'accuracy': 0.9790209790209791, 'precision': 0.967741935483871, 'recall': 1.0, 'f1': 0.9836065573770492}


,pred_0,pred_1
true_0,50,3
true_1,0,90


TN=50 (верно: 0), FP=3 (ложная тревога), FN=0 (пропуск), TP=90 (верно: 1)


array([[50,  3],
       [ 0, 90]])

## Лучший практический способ: Pipeline

Pipeline гарантирует правильный порядок:
- при `fit` scaler учится на train,
- при `predict` transform применяется к test.

Это снижает риск "случайно сделать неправильно".


In [11]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier(n_neighbors=7))
])

pipe.fit(X_train, y_train)
y_pred_pipe = pipe.predict(X_test)

pipe_metrics = summarize_binary_metrics(y_test, y_pred_pipe)
print("PIPELINE metrics:", pipe_metrics)


PIPELINE metrics: {'accuracy': 0.9790209790209791, 'precision': 0.967741935483871, 'recall': 1.0, 'f1': 0.9836065573770492}


## Итоговое сравнение и вывод

Наблюдение, которое должно "щёлкнуть":
- "плохой протокол" даёт завышение метрик,
- "хороший протокол" честнее.

Важно: если разница маленькая – это тоже урок:
- значит в этой задаче scaler не так сильно влияет,
- но правило "fit только на train" всё равно обязательно.

Запишите правило:
> Любая операция, которая настраивает параметры по данным (`fit`), должна быть внутри train‑контура.

Это один из пунктов будущего чеклиста анти‑leakage.


In [12]:
summary = pd.DataFrame([
    {"protocol": "BAD (fit scaler on all X)", **bad},
    {"protocol": "GOOD (fit scaler on train)", **good},
    {"protocol": "Pipeline", **pipe_metrics},
]).set_index("protocol")

display(summary)


,accuracy,precision,recall,f1
protocol,,,,
BAD (fit scaler on all X),0.972028,0.967391,0.988889,0.978022
GOOD (fit scaler on train),0.979021,0.967742,1.000000,0.983607
Pipeline,0.979021,0.967742,1.000000,0.983607
